# 02. Diagnóstico inicial

**Fase 2. Persona A**

Este notebook corre el diagnóstico completo sobre `establecimientos_raw.csv`
usando las funciones genéricas de `src/diagnostico.py` y el catálogo oficial
de `src/catalogos.py`. Persona C vuelve a correr estas mismas funciones sobre
el dataset limpio en la Fase 5b, para que la comparación Antes/Después sea
válida.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src import diagnostico, catalogos

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

df = pd.read_csv("../data/raw/establecimientos_raw.csv", dtype=str)

## 1. Número de registros y variables

In [2]:
print("Registros:", df.shape[0])
print("Variables:", df.shape[1])

Registros: 11891
Variables: 17


## 2. Tipo de dato de cada variable

In [3]:
diagnostico.resumen_tipos(df)

,tipo
columna,
CODIGO,str
DISTRITO,str
DEPARTAMENTO,str
MUNICIPIO,str
ESTABLECIMIENTO,str
DIRECCION,str
TELEFONO,str
SUPERVISOR,str
DIRECTOR,str


Todas las columnas quedan como texto al leer el CSV crudo, incluida `CODIGO`.
Es esperable en esta etapa. La Fase 4 decide qué columnas deben convertirse a
categórica o numérica.

## 3. Cantidad y % de valores faltantes por variable

In [4]:
diagnostico.resumen_faltantes(df)

,faltantes,porcentaje
DIRECTOR,1755,14.76
TELEFONO,969,8.15
SUPERVISOR,559,4.70
DISTRITO,556,4.68
DIRECCION,99,0.83
ESTABLECIMIENTO,28,0.24
AREA,23,0.19
PLAN,23,0.19
JORNADA,23,0.19
MODALIDAD,23,0.19


## 4. Cantidad de valores únicos por variable

In [5]:
diagnostico.contar_unicos(df)

CODIGO             11868
DISTRITO            1681
DEPARTAMENTO          23
MUNICIPIO            352
ESTABLECIMIENTO     6312
DIRECCION           7439
TELEFONO            6572
SUPERVISOR          1280
DIRECTOR            5520
NIVEL                  1
SECTOR                 4
AREA                   3
STATUS                 5
MODALIDAD              2
JORNADA                6
PLAN                  13
DEPARTAMENTAL         26
Name: valores_unicos, dtype: int64

## 5. Registros duplicados exactos

In [6]:
n_dup = diagnostico.contar_duplicados_exactos(df)
print("Duplicados exactos:", n_dup)

dup_no_vacios = diagnostico.contar_duplicados_exactos(df.dropna(subset=["CODIGO"]))
print("Duplicados exactos excluyendo filas 100% vacias:", dup_no_vacios)

Duplicados exactos: 22
Duplicados exactos excluyendo filas 100% vacias: 0


Los duplicados exactos corresponden a las filas 100% vacías que quedan como
separador al concatenar los CSV por departamento. No hay establecimientos
duplicados de verdad en el crudo.

## 6. Variables con valores fuera de dominio o inconsistentes

In [7]:
deptos_invalidos = df.loc[
    df["DEPARTAMENTO"].notna() & ~df["DEPARTAMENTO"].apply(catalogos.es_departamento_valido),
    "DEPARTAMENTO",
].value_counts()
print("Valores de DEPARTAMENTO fuera del catalogo oficial:")
deptos_invalidos

Valores de DEPARTAMENTO fuera del catalogo oficial:


DEPARTAMENTO
CIUDAD CAPITAL    2161
Name: count, dtype: int64

In [8]:
filas_validas = df.dropna(subset=["DEPARTAMENTO", "MUNICIPIO"])
municipio_valido = filas_validas.apply(
    lambda r: catalogos.es_municipio_valido(r["MUNICIPIO"], r["DEPARTAMENTO"]), axis=1
)
municipios_invalidos = filas_validas.loc[~municipio_valido, "MUNICIPIO"].value_counts()
print("Cantidad de filas con MUNICIPIO fuera de dominio:", (~municipio_valido).sum())
municipios_invalidos.head(20)

Cantidad de filas con MUNICIPIO fuera de dominio: 2362


MUNICIPIO
ZONA 1                          868
ZONA 7                          236
ZONA 12                         149
ZONA 18                         134
ZONA 2                          118
ZONA 6                           94
ZONA 11                          80
ZONA 19                          65
ZONA 3                           61
ZONA 13                          61
ZONA 10                          56
ZONA 5                           45
ZONA 21                          40
ZONA 9                           38
ZONA 17                          30
SAN MIGUEL USPANTAN              26
LA TINTA                         23
ZONA 14                          22
SANTO TOMAS CHICHICASTENANGO     21
ZONA 15                          20
Name: count, dtype: int64

In [9]:
comparables = df.dropna(subset=["DEPARTAMENTO", "DEPARTAMENTAL"])
mismatch = comparables[
    comparables["DEPARTAMENTO"].str.strip().str.upper() != comparables["DEPARTAMENTAL"].str.strip().str.upper()
]
print("Filas donde DEPARTAMENTO y DEPARTAMENTAL no coinciden:", mismatch.shape[0])
mismatch[["DEPARTAMENTO", "DEPARTAMENTAL"]].drop_duplicates()

Filas donde DEPARTAMENTO y DEPARTAMENTAL no coinciden: 6095


,DEPARTAMENTO,DEPARTAMENTAL
1324,CIUDAD CAPITAL,GUATEMALA NORTE
2377,CIUDAD CAPITAL,GUATEMALA ORIENTE
2516,CIUDAD CAPITAL,GUATEMALA OCCIDENTE
2856,CIUDAD CAPITAL,GUATEMALA SUR
4354,GUATEMALA,GUATEMALA NORTE
4370,GUATEMALA,GUATEMALA ORIENTE
4630,GUATEMALA,GUATEMALA OCCIDENTE
5421,GUATEMALA,GUATEMALA SUR
7846,PETEN,PETÉN
8915,QUICHE,QUICHÉ


## 7. Variables con formatos inconsistentes

In [10]:
telefonos = df["TELEFONO"].dropna()
no_numericos = telefonos[~telefonos.str.match(r"^\d+$")]
print("Telefonos no puramente numericos:", no_numericos.shape[0])
print("Distribucion de longitud de TELEFONO. Un telefono guatemalteco valido son 8 digitos.")
telefonos.str.len().value_counts().sort_index()

Telefonos no puramente numericos: 201
Distribucion de longitud de TELEFONO. Un telefono guatemalteco valido son 8 digitos.


TELEFONO
2         1
4         1
5         4
6         8
7        34
8     10672
9         4
10        2
11       11
13        2
14        1
15       26
16       20
17      104
18       11
19        4
20        2
23        2
25        4
26        8
30        1
Name: count, dtype: int64

In [11]:
codigos = df["CODIGO"].dropna()
codigo_mal_formado = codigos[~codigos.str.match(r"^\d{2}-\d{2}-\d{4}-\d{2}$")]
print("CODIGO que no matchea el patron NN-NN-NNNN-NN:", codigo_mal_formado.shape[0])

CODIGO que no matchea el patron NN-NN-NNNN-NN: 0


## 8. Resumen escrito de problemas por variable

Este resumen es el insumo directo del Plan de limpieza, Fase 3, Persona B.

| Variable | Problema detectado |
|---|---|
| `CODIGO` | Formato consistente `NN-NN-NNNN-NN` en el 100% de las filas no vacías. 23 faltantes, filas separadoras vacías. |
| `DISTRITO` | 556 faltantes. Formato similar a `CODIGO` de establecimiento pero sin validar. |
| `DEPARTAMENTO` | 23 faltantes. Fuera de dominio: incluye `CIUDAD CAPITAL`, que no es uno de los 22 departamentos oficiales. El buscador MINEDUC lo trata como entidad separada de `GUATEMALA`. |
| `MUNICIPIO` | 23 faltantes. Valores fuera de dominio: aparecen zonas como `ZONA 1` y `ZONA 7` en vez de un municipio real, sobre todo en filas de `CIUDAD CAPITAL`. |
| `ESTABLECIMIENTO` | 28 faltantes. Sin normalizar mayúsculas ni tildes todavía. |
| `DIRECCION` | 99 faltantes. Formato libre, sin estructura consistente. |
| `TELEFONO` | 969 faltantes, 8.15%. Formato muy inconsistente: 201 valores no puramente numéricos, con varios teléfonos concatenados con guion o diagonal, longitudes de 2 a 30 caracteres cuando lo esperable es 8 dígitos. |
| `SUPERVISOR` | 559 faltantes. Texto libre, sin normalizar. |
| `DIRECTOR` | 1,755 faltantes, 14.76%, la variable con más faltantes. Texto libre, sin normalizar. |
| `NIVEL` | 23 faltantes. Dominio único esperado, `DIVERSIFICADO`, por el filtro de descarga. Verificar que no haya otros valores colados. |
| `SECTOR` | 23 faltantes. Dominio de 4 categorías: `OFICIAL`, `PRIVADO`, `COOPERATIVA`, `MUNICIPAL`. Revisar consistencia de escritura. |
| `AREA` | 23 faltantes. Incluye una categoría `SIN ESPECIFICAR` en 3 filas además de `URBANA` y `RURAL`. |
| `STATUS` | 23 faltantes. 5 categorías: `ABIERTA`, `CERRADA TEMPORALMENTE`, `CERRADA DEFINITIVAMENTE`, `TEMPORAL TITULOS`, `TEMPORAL NOMBRAMIENTO`. Revisar consistencia. |
| `MODALIDAD` | 23 faltantes. Dominio de 2 categorías: `MONOLINGUE`, `BILINGUE`. |
| `JORNADA` | 23 faltantes. 6 categorías, sin anomalías aparentes a simple vista. |
| `PLAN` | 23 faltantes. Alta cardinalidad de categorías, 14 en total, varias parecen variantes solapadas: `SEMIPRESENCIAL`, `SEMIPRESENCIAL (FIN DE SEMANA)`, `SEMIPRESENCIAL (UN DIA A LA SEMANA)`, `SEMIPRESENCIAL (DOS DIAS A LA SEMANA)`. Candidato a revisar consistencia de categorías. |
| `DEPARTAMENTAL` | 23 faltantes. Inconsistente con `DEPARTAMENTO` en 6,095 filas: trae subregiones como `GUATEMALA NORTE`, `GUATEMALA SUR`, `GUATEMALA ORIENTE`, `GUATEMALA OCCIDENTE`, y nombres con tilde como `QUICHE`, `SOLOLA`, `PETEN`, `SACATEPEQUEZ`, `SUCHITEPEQUEZ`, `TOTONICAPAN`. Requiere decidir en Fase 3 si se conserva, se deriva de `DEPARTAMENTO` o se elimina. |

23 filas están 100% vacías. Son separadoras de la concatenación por
departamento en Fase 1, deben eliminarse en la limpieza, y son la causa de
los 23 faltantes que se repiten en casi todas las columnas y de los 22
duplicados exactos.